In [1]:
!pip install roboflow
!pip install ultralytics



In [4]:
from roboflow import Roboflow
size = 320

In [ ]:
rf = Roboflow(api_key="your-roboflow-apikey")  # Replace with your Roboflow API key
project = rf.workspace("your-workspace-id").project("your project id")
version = project.version(1)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...


In [6]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 20.6/107.7 GB disk)


In [7]:
from ultralytics import YOLO
from IPython.display import Image
dataset.location

'/content/Mouse-1'

In [ ]:
!yolo task=detect mode=train data={dataset.location}/data.yaml model="yolo11n.pt" epochs=50 imgsz={size}

Ultralytics 8.4.92 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Mouse-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tr

### 使用自定义模型在图像上进行推理

In [ ]:
!yolo task=detect mode=val model="/content/runs/detect/train/weights/best.pt" data={dataset.location}/data.yaml

### 模型转换

In [ ]:
# 1. 安装 .NET 7.0 运行时 (nncase 依赖)
!sudo apt-get update
!sudo apt-get install -y dotnet-sdk-7.0

# 2. 安装 nncase 和 nncase-kpu (以 2.9.0 版本为例)
!pip install nncase==2.9.0 nncase-kpu==2.9.0

# 3. 安装 ONNX 相关依赖
!pip install onnx onnxruntime onnxsim

In [ ]:
from ultralytics import YOLO

# 加载你训练好的模型文件
model = YOLO("/content/runs/detect/train/weights/best.pt")

# 导出为 ONNX 格式
model.export(
    format="onnx",
    imgsz=(320, 320),  # 必须与你训练和后续转换的尺寸一致
    optimize=True,     # 进行优化
    simplify=True      # 简化模型结构
)

In [ ]:
# 1. 获取官方转换工具 (如果还没有)
!git clone https://github.com/kendryte/k230_training_scripts.git

# 2. 进入检测模型转换脚本目录
%cd k230_training_scripts/yolo_files/detect/


In [ ]:

# 3. 执行转换命令
!python to_kmodel.py \
    --model /content/runs/detect/train/weights/best.onnx \
    --dataset {dataset.location}/train/images \
    --input_width {size} \
    --input_height {size}